# U.S. Household Financial Pressure Dashboard

This notebook uses the existing local FRED CSV files in `data/` to answer: **Are U.S. households under financial pressure right now, and which indicators are driving that pressure?**

The core household financial pressure score is unchanged. It uses only Debt Service Ratio, Personal Saving Rate, and Credit Card Delinquency Rate. All other series are context indicators only.

In [77]:
import pandas as pd
import altair as alt
from IPython.display import display
from pathlib import Path

alt.data_transformers.disable_max_rows()

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "html_charts"
OUTPUT_DIR.mkdir(exist_ok=True)

SERIES_CONFIG = {
    "TDSP": {"file": "TDSP.csv", "value_col": "TDSP", "indicator": "Debt Service Ratio", "category": "Core household", "direction": "higher_is_worse"},
    "PSAVERT": {"file": "PSAVERT.csv", "value_col": "PSAVERT", "indicator": "Personal Saving Rate", "category": "Core household", "direction": "lower_is_worse"},
    "DRCCLACBS": {"file": "DRCCLACBS.csv", "value_col": "DRCCLACBS", "indicator": "Credit Card Delinquency Rate", "category": "Core household", "direction": "higher_is_worse"},
    "UNRATE": {"file": "UNRATE.csv", "value_col": "UNRATE", "indicator": "Unemployment Rate", "category": "Economic context", "direction": "higher_is_worse"},
    "FEDFUNDS": {"file": "FEDFUNDS.csv", "value_col": "FEDFUNDS", "indicator": "Federal Funds Rate", "category": "Economic context", "direction": "higher_is_worse"},
}

INDICATOR_ORDER = [
    "Debt Service Ratio",
    "Personal Saving Rate",
    "Credit Card Delinquency Rate",
    "Unemployment Rate",
    "Federal Funds Rate",
]
CORE_INDICATORS = ["Debt Service Ratio", "Personal Saving Rate", "Credit Card Delinquency Rate"]
PRESSURE_COLORS = ["#2a9d8f", "#f4a261", "#d95f5f"]
GROUP_COLORS = ["#33658a", "#9b5de5", "#2a9d8f"]


def clean_local_fred_csv(file_name, value_col):
    df = pd.read_csv(DATA_DIR / file_name)
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    return df.dropna(subset=[value_col]).sort_values("observation_date").reset_index(drop=True)


def clean_configured_series(series_code):
    config = SERIES_CONFIG[series_code]
    return clean_local_fred_csv(config["file"], config["value_col"])


series_data = {series_code: clean_configured_series(series_code) for series_code in SERIES_CONFIG}

data_inventory = pd.DataFrame([
    {
        "Series": config["indicator"],
        "FRED code": series_code,
        "Rows": len(series_data[series_code]),
        "First date": series_data[series_code]["observation_date"].min().strftime("%Y-%m-%d"),
        "Latest date": series_data[series_code]["observation_date"].max().strftime("%Y-%m-%d"),
    }
    for series_code, config in SERIES_CONFIG.items()
])

display(data_inventory)


,Series,FRED code,Rows,First date,Latest date
0,Debt Service Ratio,TDSP,84,2005-01-01,2025-10-01
1,Personal Saving Rate,PSAVERT,808,1959-01-01,2026-04-01
2,Credit Card Delinquency Rate,DRCCLACBS,141,1991-01-01,2026-01-01
3,Unemployment Rate,UNRATE,940,1948-01-01,2026-05-01
4,Federal Funds Rate,FEDFUNDS,863,1954-07-01,2026-05-01


In [78]:
def pressure_status(score):
    if score >= 75:
        return "elevated"
    if score >= 40:
        return "moderate/mixed"
    return "lower"


def direction_label(direction):
    return "higher is worse" if direction == "higher_is_worse" else "lower is worse"


def pressure_percentile_for_values(values, latest_value, direction):
    historical_percentile = (values <= latest_value).mean() * 100
    if direction == "higher_is_worse":
        return historical_percentile
    return 100 - historical_percentile


def summarize_indicator(series_code):
    config = SERIES_CONFIG[series_code]
    value_col = config["value_col"]
    df = series_data[series_code]
    latest_row = df.iloc[-1]
    latest_value = latest_row[value_col]
    values = df[value_col].dropna()
    pressure_percentile = pressure_percentile_for_values(values, latest_value, config["direction"])
    return {
        "series_code": series_code,
        "indicator": config["indicator"],
        "group": config["category"],
        "latest_date": latest_row["observation_date"],
        "latest_value": latest_value,
        "unit": "%",
        "historical_average": values.mean(),
        "historical_median": values.median(),
        "historical_min": values.min(),
        "historical_max": values.max(),
        "pressure_percentile": pressure_percentile,
        "pressure_level": pressure_status(pressure_percentile),
        "pressure_direction": config["direction"],
        "pressure_direction_label": direction_label(config["direction"]),
        "indicator_order": INDICATOR_ORDER.index(config["indicator"]),
    }


summary = pd.DataFrame([summarize_indicator(series_code) for series_code in SERIES_CONFIG])
summary = summary.sort_values("indicator_order").reset_index(drop=True)
summary["latest_date_label"] = summary["latest_date"].dt.strftime("%Y-%m-%d")
summary["latest_value_label"] = summary["latest_value"].map(lambda value: f"{value:.2f}%")
summary["pressure_percentile_label"] = summary["pressure_percentile"].map(lambda value: f"{value:.1f}")

core_summary = summary[summary["indicator"].isin(CORE_INDICATORS)].copy()
overall_pressure_score = core_summary["pressure_percentile"].mean()
overall_status = pressure_status(overall_pressure_score)
latest_date_used = core_summary["latest_date"].max()

scorecard_display = pd.concat([
    pd.DataFrame([{
        "Metric": "Overall household financial pressure score",
        "Latest date": f"Through {latest_date_used:%Y-%m-%d}*",
        "Latest value": f"{overall_pressure_score:.1f} / 100",
        "Pressure percentile": f"{overall_pressure_score:.1f}",
        "Status": overall_status,
    }]),
    core_summary.assign(
        Metric=core_summary["indicator"],
        **{
            "Latest date": core_summary["latest_date_label"],
            "Latest value": core_summary["latest_value_label"],
            "Pressure percentile": core_summary["pressure_percentile_label"],
            "Status": core_summary["pressure_level"],
        },
    )[["Metric", "Latest date", "Latest value", "Pressure percentile", "Status"]],
], ignore_index=True)


## 1. Overview / Scorecard

The scorecard averages the normalized pressure scores for the three core household indicators: debt service, personal saving, and credit card delinquency. `*` The score uses each core series' latest available observation, so individual indicator dates differ by release schedule.

In [79]:
display(scorecard_display.rename(columns={"Pressure percentile": "Normalized pressure score"}))


,Metric,Latest date,Latest value,Normalized pressure score,Status
0,Overall household financial pressure score,Through 2026-04-01*,52.1 / 100,52.1,moderate/mixed
1,Debt Service Ratio,2025-10-01,11.32%,27.4,lower
2,Personal Saving Rate,2026-04-01,2.60%,96.4,elevated
3,Credit Card Delinquency Rate,2026-01-01,2.92%,32.6,lower


## 2. Main Pressure Score Chart

The normalized score converts each original indicator to a 0-100 normalized pressure score. Higher bars mean the latest reading is more concerning relative to that indicator's own history. The personal saving rate is reversed because lower saving signals more pressure.

In [80]:
pressure_tooltip = [
    alt.Tooltip("indicator:N", title="Indicator"),
    alt.Tooltip("group:N", title="Role"),
    alt.Tooltip("latest_value:Q", title="Latest value (%)", format=".2f"),
    alt.Tooltip("latest_date:T", title="Latest date", format="%Y-%m-%d"),
    alt.Tooltip("pressure_percentile:Q", title="Normalized pressure score", format=".1f"),
    alt.Tooltip("pressure_level:N", title="Pressure level"),
]

pressure_base = alt.Chart(summary).encode(
    x=alt.X("pressure_percentile:Q", title="Normalized pressure score: 0 = low concern, 100 = high concern", scale=alt.Scale(domain=[0, 100])),
    y=alt.Y("indicator:N", title=None, sort=alt.SortField(field="pressure_percentile", order="descending")),
)
pressure_bars = pressure_base.mark_bar(cornerRadiusEnd=3).encode(
    color=alt.Color("pressure_percentile:Q", title="Higher = more pressure", scale=alt.Scale(domain=[0, 50, 100], range=PRESSURE_COLORS)),
    tooltip=pressure_tooltip,
)
pressure_labels = pressure_base.mark_text(align="right", baseline="middle", dx=-6, color="white", fontWeight="bold").encode(
    text=alt.Text("pressure_percentile:Q", format=".1f")
)
pressure_score_chart = (pressure_bars + pressure_labels).properties(width=430, height=250)
display(pressure_score_chart)


alt.LayerChart(...)

## 3. Core Household Indicator Time Series

Each core chart shows the full historical series, a dashed historical average, and a point for the latest observation.

In [81]:
def make_indicator_line_chart(series_code, title, y_title, line_color="#33658a"):
    config = SERIES_CONFIG[series_code]
    value_col = config["value_col"]
    df = series_data[series_code]
    latest = df.tail(1)
    average_df = pd.DataFrame({"historical_average": [df[value_col].mean()]})

    line = alt.Chart(df).mark_line(color=line_color, strokeWidth=2).encode(
        x=alt.X("observation_date:T", title="Date"),
        y=alt.Y(f"{value_col}:Q", title=y_title, scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("observation_date:T", title="Date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Value (%)", format=".2f")],
    )
    average_rule = alt.Chart(average_df).mark_rule(color="#6c757d", strokeDash=[6, 4], strokeWidth=1.5).encode(
        y=alt.Y("historical_average:Q"),
        tooltip=[alt.Tooltip("historical_average:Q", title="Historical average (%)", format=".2f")],
    )
    latest_point = alt.Chart(latest).mark_circle(size=95, color="#d95f5f", stroke="white", strokeWidth=1).encode(
        x=alt.X("observation_date:T"),
        y=alt.Y(f"{value_col}:Q"),
        tooltip=[alt.Tooltip("observation_date:T", title="Latest date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Latest value (%)", format=".2f")],
    )
    return (line + average_rule + latest_point).properties(title=title, width=900, height=240)


### Debt Service Ratio

Debt service shows how much household disposable income goes toward required debt payments. Higher readings leave households with less room to absorb shocks or rising costs.

In [82]:
debt_service_chart = make_indicator_line_chart("TDSP", "Debt Service Ratio", "Debt Service Ratio (%)", line_color="#33658a")
display(debt_service_chart)


alt.LayerChart(...)

### Personal Saving Rate

The personal saving rate is a cushion indicator. Lower saving can mean households have less cash buffer after spending, so low readings increase the pressure score.

In [83]:
saving_rate_chart = make_indicator_line_chart("PSAVERT", "Personal Saving Rate", "Personal Saving Rate (%)", line_color="#2a9d8f")
display(saving_rate_chart)


alt.LayerChart(...)

### Credit Card Delinquency Rate

Credit card delinquency is a direct sign of repayment stress. Rising delinquency suggests more households are falling behind on short-term consumer debt.

In [84]:
delinquency_chart = make_indicator_line_chart("DRCCLACBS", "Credit Card Delinquency Rate", "Credit Card Delinquency Rate (%)", line_color="#9b5de5")
display(delinquency_chart)


alt.LayerChart(...)

## 4. Economic Context Charts

Unemployment and the federal funds rate are context indicators. They help explain the environment around households, but they are **not** averaged into the core household pressure score.

In [85]:
def make_context_chart(series_code, title, y_title, line_color):
    config = SERIES_CONFIG[series_code]
    value_col = config["value_col"]
    df = series_data[series_code]
    latest = df.tail(1)
    line = alt.Chart(df).mark_line(color=line_color, strokeWidth=2).encode(
        x=alt.X("observation_date:T", title="Date"),
        y=alt.Y(f"{value_col}:Q", title=y_title, scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("observation_date:T", title="Date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Value (%)", format=".2f")],
    )
    latest_point = alt.Chart(latest).mark_circle(size=90, color="#d95f5f", stroke="white", strokeWidth=1).encode(
        x=alt.X("observation_date:T"), y=alt.Y(f"{value_col}:Q"),
        tooltip=[alt.Tooltip("observation_date:T", title="Latest date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Latest value (%)", format=".2f")],
    )
    return (line + latest_point).properties(title=title, width=430, height=230)

unemployment_chart = make_context_chart("UNRATE", "Unemployment Rate", "Unemployment Rate (%)", line_color="#33658a")
fed_funds_chart = make_context_chart("FEDFUNDS", "Federal Funds Rate", "Federal Funds Rate (%)", line_color="#9b5de5")
economic_context_charts = alt.hconcat(unemployment_chart, fed_funds_chart).resolve_scale(y="independent")
display(economic_context_charts)


alt.HConcatChart(...)

## Additional Household Pressure Context

The next indicators broaden the household pressure story without changing the original core score. Inflation, real earnings, sentiment, and housing affordability help explain why households may feel more or less pressure even when the three direct household indicators are mixed.

In [86]:
def make_latest_line_chart(df, value_col, title, y_title, line_color="#33658a", value_format=".2f"):
    latest = df.tail(1)
    line = alt.Chart(df).mark_line(color=line_color, strokeWidth=2).encode(
        x=alt.X("observation_date:T", title="Date"),
        y=alt.Y(f"{value_col}:Q", title=y_title, scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("observation_date:T", title="Date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title=y_title, format=value_format)],
    )
    latest_point = alt.Chart(latest).mark_circle(size=90, color="#d95f5f", stroke="white", strokeWidth=1).encode(
        x=alt.X("observation_date:T"), y=alt.Y(f"{value_col}:Q"),
        tooltip=[alt.Tooltip("observation_date:T", title="Latest date", format="%Y-%m-%d"), alt.Tooltip(f"{value_col}:Q", title="Latest value", format=value_format)],
    )
    return (line + latest_point).properties(title=title, width=430, height=230)


def summarize_context_indicator(df, value_col, indicator, group, unit, direction):
    clean_df = df.dropna(subset=[value_col]).sort_values("observation_date")
    latest_row = clean_df.iloc[-1]
    values = clean_df[value_col].dropna()
    pressure_percentile = pressure_percentile_for_values(values, latest_row[value_col], direction)
    return {
        "indicator": indicator,
        "group": group,
        "latest_date": latest_row["observation_date"],
        "latest_value": latest_row[value_col],
        "unit": unit,
        "historical_median": values.median(),
        "pressure_percentile": pressure_percentile,
        "pressure_level": pressure_status(pressure_percentile),
        "pressure_direction": direction,
        "pressure_direction_label": direction_label(direction),
    }


### Inflation / CPI Context

Year-over-year CPI inflation captures cost-of-living pressure. Higher inflation can squeeze household budgets even when income or debt measures look stable.

In [87]:
cpi = clean_local_fred_csv("CPIAUCSL.csv", "CPIAUCSL")
cpi["cpi_yoy"] = cpi["CPIAUCSL"].pct_change(12) * 100
cpi_yoy = cpi.dropna(subset=["cpi_yoy"]).reset_index(drop=True)

inflation_cpi_yoy_chart = make_latest_line_chart(
    cpi_yoy, "cpi_yoy", "Inflation Pressure: CPI Year-over-Year Change", "CPI year-over-year change (%)", line_color="#d95f5f"
)
display(inflation_cpi_yoy_chart)


alt.LayerChart(...)

### Real Weekly Earnings Context

Real median weekly earnings show whether inflation-adjusted wages are improving or weakening. Lower real earnings count as more pressure in the expanded context logic.

In [88]:
real_weekly_earnings = clean_local_fred_csv("LES1252881600Q.csv", "LES1252881600Q")
real_weekly_earnings_chart = make_latest_line_chart(
    real_weekly_earnings, "LES1252881600Q", "Real Median Weekly Earnings", "Real median weekly earnings", line_color="#2a9d8f", value_format=",.0f"
)
display(real_weekly_earnings_chart)


alt.LayerChart(...)

### Consumer Sentiment Context

Consumer sentiment captures how households feel about the economy. Lower sentiment can signal more anxiety about income, prices, employment, or future finances.

In [89]:
consumer_sentiment = clean_local_fred_csv("UMCSENT.csv", "UMCSENT")
consumer_sentiment_chart = make_latest_line_chart(
    consumer_sentiment, "UMCSENT", "Consumer Sentiment", "Consumer sentiment index", line_color="#33658a", value_format=".1f"
)
display(consumer_sentiment_chart)


alt.LayerChart(...)

### Housing Affordability Context

This approximation estimates the monthly principal-and-interest payment on the median home sale price using 20% down and a 30-year fixed mortgage, then compares that payment with monthly real median household income. It is a housing affordability pressure context measure only and is not part of the core household score.

In [90]:
home_prices = clean_local_fred_csv("MSPUS.csv", "MSPUS")
mortgage_rates = clean_local_fred_csv("MORTGAGE30US.csv", "MORTGAGE30US")
household_income = clean_local_fred_csv("MEHOINUSA672N.csv", "MEHOINUSA672N")

home_prices_q = home_prices.set_index("observation_date")[["MSPUS"]].resample("QE").mean()
mortgage_rates_q = mortgage_rates.set_index("observation_date")[["MORTGAGE30US"]].resample("QE").mean()
household_income_q = household_income.set_index("observation_date")[["MEHOINUSA672N"]].resample("QE").ffill()

housing_affordability = home_prices_q.join(mortgage_rates_q, how="outer").join(household_income_q, how="outer")
housing_affordability["MEHOINUSA672N"] = housing_affordability["MEHOINUSA672N"].ffill()
housing_affordability = housing_affordability.dropna(subset=["MSPUS", "MORTGAGE30US", "MEHOINUSA672N"]).reset_index()

loan_amount = housing_affordability["MSPUS"] * 0.80
monthly_rate = housing_affordability["MORTGAGE30US"] / 100 / 12
number_of_payments = 30 * 12
housing_affordability["estimated_monthly_payment"] = loan_amount * (
    monthly_rate * (1 + monthly_rate) ** number_of_payments
) / ((1 + monthly_rate) ** number_of_payments - 1)
housing_affordability["monthly_household_income"] = housing_affordability["MEHOINUSA672N"] / 12
housing_affordability["payment_to_income_pct"] = housing_affordability["estimated_monthly_payment"] / housing_affordability["monthly_household_income"] * 100

housing_y_max = max(45, housing_affordability["payment_to_income_pct"].max() * 1.08)
housing_bands = pd.DataFrame([
    {"band": "Under 25%: more affordable", "start": housing_affordability["observation_date"].min(), "end": housing_affordability["observation_date"].max(), "y_min": 0, "y_max": 25},
    {"band": "25-35%: strained", "start": housing_affordability["observation_date"].min(), "end": housing_affordability["observation_date"].max(), "y_min": 25, "y_max": 35},
    {"band": "Above 35%: highly strained", "start": housing_affordability["observation_date"].min(), "end": housing_affordability["observation_date"].max(), "y_min": 35, "y_max": housing_y_max},
])

housing_band_layer = alt.Chart(housing_bands).mark_rect(opacity=0.22).encode(
    x=alt.X("start:T", title="Date"), x2="end:T",
    y=alt.Y("y_min:Q", title="Payment as share of monthly household income (%)"), y2="y_max:Q",
    color=alt.Color("band:N", title="Interpretation band", scale=alt.Scale(range=["#2a9d8f", "#f4a261", "#d95f5f"])),
    tooltip=[alt.Tooltip("band:N", title="Band")],
)
housing_line = alt.Chart(housing_affordability).mark_line(color="#263238", strokeWidth=2).encode(
    x=alt.X("observation_date:T", title="Date"),
    y=alt.Y("payment_to_income_pct:Q", title="Payment as share of monthly household income (%)", scale=alt.Scale(domain=[0, housing_y_max])),
    tooltip=[
        alt.Tooltip("observation_date:T", title="Quarter", format="%Y-%m-%d"),
        alt.Tooltip("MSPUS:Q", title="Median home sale price", format=",.0f"),
        alt.Tooltip("MORTGAGE30US:Q", title="30-year mortgage rate (%)", format=".2f"),
        alt.Tooltip("MEHOINUSA672N:Q", title="Annual median household income", format=",.0f"),
        alt.Tooltip("estimated_monthly_payment:Q", title="Estimated monthly payment", format=",.0f"),
        alt.Tooltip("payment_to_income_pct:Q", title="Payment-to-income (%)", format=".1f"),
    ],
)
housing_latest_point = alt.Chart(housing_affordability.tail(1)).mark_circle(size=90, color="#d95f5f", stroke="white", strokeWidth=1).encode(
    x=alt.X("observation_date:T"), y=alt.Y("payment_to_income_pct:Q"),
    tooltip=[alt.Tooltip("observation_date:T", title="Latest quarter", format="%Y-%m-%d"), alt.Tooltip("payment_to_income_pct:Q", title="Latest payment-to-income (%)", format=".1f")],
)
housing_affordability_chart = (housing_band_layer + housing_line + housing_latest_point).properties(
    title="Estimated Mortgage Payment as Share of Household Income", width=430, height=230
)
display(housing_affordability_chart)


alt.LayerChart(...)

In [91]:
additional_household_pressure_context = alt.vconcat(
    alt.hconcat(inflation_cpi_yoy_chart, real_weekly_earnings_chart).resolve_scale(y="independent"),
    alt.hconcat(consumer_sentiment_chart, housing_affordability_chart).resolve_scale(y="independent"),
    spacing=20,
).properties(title="Additional Household Pressure Context").resolve_scale(y="independent")
display(additional_household_pressure_context)


alt.VConcatChart(...)

## Household Financial Pressure Scores Across All Indicators

Latest normalized pressure scores across core and context indicators, showing which areas are contributing the most and least to household financial strain.

Each dot shows where the latest reading sits relative to that indicator's own history after adjusting direction so higher scores mean more household pressure. Context indicators help explain the environment but do not change the core household pressure score.

In [92]:
context_pressure_rows = pd.DataFrame([
    summarize_context_indicator(cpi_yoy, "cpi_yoy", "CPI YoY Inflation", "Additional context", "%", "higher_is_worse"),
    summarize_context_indicator(real_weekly_earnings, "LES1252881600Q", "Real Median Weekly Earnings", "Additional context", "real dollars", "lower_is_worse"),
    summarize_context_indicator(consumer_sentiment, "UMCSENT", "Consumer Sentiment", "Additional context", "index", "lower_is_worse"),
    summarize_context_indicator(housing_affordability, "payment_to_income_pct", "Housing Payment-to-Income Ratio", "Additional context", "%", "higher_is_worse"),
])

latest_pressure_percentiles = pd.concat([
    summary[[
        "indicator", "group", "latest_date", "latest_value", "unit", "historical_median",
        "pressure_percentile", "pressure_level", "pressure_direction", "pressure_direction_label"
    ]],
    context_pressure_rows,
], ignore_index=True)

latest_pressure_order = [
    "Debt Service Ratio",
    "Personal Saving Rate",
    "Credit Card Delinquency Rate",
    "Unemployment Rate",
    "Federal Funds Rate",
    "CPI YoY Inflation",
    "Real Median Weekly Earnings",
    "Consumer Sentiment",
    "Housing Payment-to-Income Ratio",
]
latest_pressure_percentiles["indicator"] = pd.Categorical(
    latest_pressure_percentiles["indicator"], categories=latest_pressure_order, ordered=True
)
latest_pressure_percentiles = latest_pressure_percentiles.sort_values("indicator").reset_index(drop=True)
latest_pressure_percentiles["latest_date_label"] = latest_pressure_percentiles["latest_date"].dt.strftime("%Y-%m-%d")

median_reference = pd.DataFrame({"pressure_percentile": [50]})
median_line = alt.Chart(median_reference).mark_rule(color="#6c757d", strokeDash=[5, 5], strokeWidth=1.5).encode(
    x=alt.X("pressure_percentile:Q")
)

percentile_dots = alt.Chart(latest_pressure_percentiles).mark_circle(size=130, stroke="white", strokeWidth=1).encode(
    x=alt.X("pressure_percentile:Q", title="Normalized pressure score: 0 = low concern, 100 = high concern", scale=alt.Scale(domain=[0, 100])),
    y=alt.Y("indicator:N", title=None, sort=latest_pressure_order),
    color=alt.Color(
        "group:N",
        title="Indicator group",
        scale=alt.Scale(domain=["Core household", "Economic context", "Additional context"], range=GROUP_COLORS),
    ),
    tooltip=[
        alt.Tooltip("indicator:N", title="Indicator"),
        alt.Tooltip("group:N", title="Group"),
        alt.Tooltip("latest_date_label:N", title="Latest date"),
        alt.Tooltip("latest_value:Q", title="Latest raw value", format=".2f"),
        alt.Tooltip("unit:N", title="Unit"),
        alt.Tooltip("historical_median:Q", title="Historical median", format=".2f"),
        alt.Tooltip("pressure_percentile:Q", title="Normalized pressure score", format=".1f"),
        alt.Tooltip("pressure_direction_label:N", title="Pressure direction"),
    ],
)

percentile_labels = alt.Chart(latest_pressure_percentiles).mark_text(align="left", baseline="middle", dx=8, color="#263238").encode(
    x=alt.X("pressure_percentile:Q"),
    y=alt.Y("indicator:N", sort=latest_pressure_order),
    text=alt.Text("pressure_percentile:Q", format=".1f"),
)

latest_pressure_percentiles_chart = (median_line + percentile_dots + percentile_labels).properties(
    width=900,
    height=340,
)

display(latest_pressure_percentiles_chart)


alt.LayerChart(...)


## Core and Expanded Stress Flags Timelines

The core chart keeps the original 0-3 calculation and uses only debt service, personal saving, and credit card delinquency. Because the three core series do not all start at the same time, the timeline now uses the full available quarterly history and includes an available-indicator count in the tooltip.

The expanded chart adds four context flags for a 0-7 view. Both timelines are displayed across the full available stress-window from the 1950s through the latest available 2026 data, instead of only showing the period where all three core indicators overlap.


In [93]:

tdsp_q = series_data["TDSP"].set_index("observation_date")[["TDSP"]].resample("QE").mean()
psavert_q = series_data["PSAVERT"].set_index("observation_date")[["PSAVERT"]].resample("QE").mean()
drcc_q = series_data["DRCCLACBS"].set_index("observation_date")[["DRCCLACBS"]].resample("QE").mean()

core_stress_q = (
    tdsp_q.rename(columns={"TDSP": "debt_service_ratio"})
    .join(psavert_q.rename(columns={"PSAVERT": "personal_saving_rate"}), how="outer")
    .join(drcc_q.rename(columns={"DRCCLACBS": "credit_card_delinquency_rate"}), how="outer")
    .reset_index()
    .sort_values("observation_date")
)
core_value_columns = ["debt_service_ratio", "personal_saving_rate", "credit_card_delinquency_rate"]
core_stress_q["available_indicator_count"] = core_stress_q[core_value_columns].notna().sum(axis=1)
core_stress_q = core_stress_q[core_stress_q["available_indicator_count"] > 0].copy()
core_stress_q["debt_service_stress"] = core_stress_q["debt_service_ratio"] > core_stress_q["debt_service_ratio"].median(skipna=True)
core_stress_q["saving_rate_stress"] = core_stress_q["personal_saving_rate"] < core_stress_q["personal_saving_rate"].median(skipna=True)
core_stress_q["credit_card_stress"] = core_stress_q["credit_card_delinquency_rate"] > core_stress_q["credit_card_delinquency_rate"].median(skipna=True)
core_flag_columns = ["debt_service_stress", "saving_rate_stress", "credit_card_stress"]
core_stress_q["stress_count"] = core_stress_q[core_flag_columns].sum(axis=1)
core_stress_q["quarter"] = core_stress_q["observation_date"].dt.to_period("Q").astype(str)

cpi_yoy_q = cpi_yoy.set_index("observation_date")[["cpi_yoy"]].resample("QE").mean()
earnings_q = real_weekly_earnings.set_index("observation_date")[["LES1252881600Q"]].resample("QE").mean()
sentiment_q = consumer_sentiment.set_index("observation_date")[["UMCSENT"]].resample("QE").mean()
housing_q = housing_affordability.set_index("observation_date")[["payment_to_income_pct"]].resample("QE").mean()

expanded_stress_q = (
    tdsp_q.rename(columns={"TDSP": "debt_service_ratio"})
    .join(psavert_q.rename(columns={"PSAVERT": "personal_saving_rate"}), how="outer")
    .join(drcc_q.rename(columns={"DRCCLACBS": "credit_card_delinquency_rate"}), how="outer")
    .join(cpi_yoy_q.rename(columns={"cpi_yoy": "cpi_yoy_inflation"}), how="outer")
    .join(earnings_q.rename(columns={"LES1252881600Q": "real_weekly_earnings"}), how="outer")
    .join(sentiment_q.rename(columns={"UMCSENT": "consumer_sentiment"}), how="outer")
    .join(housing_q, how="outer")
    .reset_index()
    .sort_values("observation_date")
)
expanded_value_columns = [
    "debt_service_ratio", "personal_saving_rate", "credit_card_delinquency_rate", "cpi_yoy_inflation",
    "real_weekly_earnings", "consumer_sentiment", "payment_to_income_pct",
]
expanded_stress_q["available_indicator_count"] = expanded_stress_q[expanded_value_columns].notna().sum(axis=1)
expanded_stress_q = expanded_stress_q[expanded_stress_q["available_indicator_count"] > 0].copy()
expanded_stress_q["debt_service_stress"] = expanded_stress_q["debt_service_ratio"] > expanded_stress_q["debt_service_ratio"].median(skipna=True)
expanded_stress_q["saving_rate_stress"] = expanded_stress_q["personal_saving_rate"] < expanded_stress_q["personal_saving_rate"].median(skipna=True)
expanded_stress_q["credit_card_stress"] = expanded_stress_q["credit_card_delinquency_rate"] > expanded_stress_q["credit_card_delinquency_rate"].median(skipna=True)
expanded_stress_q["inflation_stress"] = expanded_stress_q["cpi_yoy_inflation"] > expanded_stress_q["cpi_yoy_inflation"].median(skipna=True)
expanded_stress_q["earnings_stress"] = expanded_stress_q["real_weekly_earnings"] < expanded_stress_q["real_weekly_earnings"].median(skipna=True)
expanded_stress_q["sentiment_stress"] = expanded_stress_q["consumer_sentiment"] < expanded_stress_q["consumer_sentiment"].median(skipna=True)
expanded_stress_q["housing_stress"] = expanded_stress_q["payment_to_income_pct"] > expanded_stress_q["payment_to_income_pct"].median(skipna=True)
expanded_flag_columns = ["debt_service_stress", "saving_rate_stress", "credit_card_stress", "inflation_stress", "earnings_stress", "sentiment_stress", "housing_stress"]
expanded_stress_q["expanded_stress_count"] = expanded_stress_q[expanded_flag_columns].sum(axis=1)
expanded_stress_q["quarter"] = expanded_stress_q["observation_date"].dt.to_period("Q").astype(str)

stress_display_start = max(
    min(core_stress_q["observation_date"].min(), expanded_stress_q["observation_date"].min()),
    pd.Timestamp("1950-01-01"),
)
stress_display_end = max(core_stress_q["observation_date"].max(), expanded_stress_q["observation_date"].max())
stress_x_domain = [stress_display_start, stress_display_end]
core_stress_display_q = core_stress_q[
    (core_stress_q["observation_date"] >= stress_display_start)
    & (core_stress_q["observation_date"] <= stress_display_end)
].copy()
expanded_stress_display_q = expanded_stress_q[
    (expanded_stress_q["observation_date"] >= stress_display_start)
    & (expanded_stress_q["observation_date"] <= stress_display_end)
].copy()

core_stress_timeline_chart = alt.Chart(core_stress_display_q).mark_bar().encode(
    x=alt.X("observation_date:T", title="Quarter", scale=alt.Scale(domain=stress_x_domain), axis=alt.Axis(format="%Y")),
    y=alt.Y("stress_count:Q", title="Number of core stress flags", scale=alt.Scale(domain=[0, 3]), axis=alt.Axis(tickMinStep=1)),
    color=alt.Color("stress_count:Q", title="Stress flags", scale=alt.Scale(domain=[0, 1, 2, 3], range=["#d7ece8", "#8ecae6", "#f4a261", "#d95f5f"])),
    tooltip=[
        alt.Tooltip("quarter:N", title="Quarter"),
        alt.Tooltip("stress_count:Q", title="Number of stress flags", format=".0f"),
        alt.Tooltip("available_indicator_count:Q", title="Available core indicators", format=".0f"),
        alt.Tooltip("debt_service_ratio:Q", title="Debt Service Ratio (%)", format=".2f"),
        alt.Tooltip("personal_saving_rate:Q", title="Personal Saving Rate (%)", format=".2f"),
        alt.Tooltip("credit_card_delinquency_rate:Q", title="Credit Card Delinquency Rate (%)", format=".2f"),
    ],
).properties(width=900, height=220)

expanded_stress_timeline_chart = alt.Chart(expanded_stress_display_q).mark_bar().encode(
    x=alt.X("observation_date:T", title="Quarter", scale=alt.Scale(domain=stress_x_domain), axis=alt.Axis(format="%Y")),
    y=alt.Y("expanded_stress_count:Q", title="Number of expanded pressure flags", scale=alt.Scale(domain=[0, 7]), axis=alt.Axis(tickMinStep=1)),
    color=alt.Color("expanded_stress_count:Q", title="Pressure flags", scale=alt.Scale(domain=[0, 3.5, 7], range=["#d7ece8", "#f4a261", "#d95f5f"])),
    tooltip=[
        alt.Tooltip("quarter:N", title="Quarter"),
        alt.Tooltip("expanded_stress_count:Q", title="Expanded stress count", format=".0f"),
        alt.Tooltip("available_indicator_count:Q", title="Available indicator count", format=".0f"),
        alt.Tooltip("debt_service_ratio:Q", title="Debt Service Ratio (%)", format=".2f"),
        alt.Tooltip("personal_saving_rate:Q", title="Personal Saving Rate (%)", format=".2f"),
        alt.Tooltip("credit_card_delinquency_rate:Q", title="Credit Card Delinquency Rate (%)", format=".2f"),
        alt.Tooltip("cpi_yoy_inflation:Q", title="CPI YoY Inflation (%)", format=".2f"),
        alt.Tooltip("real_weekly_earnings:Q", title="Real Weekly Earnings", format=",.0f"),
        alt.Tooltip("consumer_sentiment:Q", title="Consumer Sentiment", format=".1f"),
        alt.Tooltip("payment_to_income_pct:Q", title="Mortgage Payment / Income (%)", format=".1f"),
    ],
).properties(width=900, height=240)

display(core_stress_timeline_chart)
display(expanded_stress_timeline_chart)


alt.Chart(...)

alt.Chart(...)

## Pressure Score Logic

`Raw FRED CSV files` -> `clean dates and values` -> `calculate historical percentiles` -> `adjust direction of pressure` -> `average core household indicators` -> `overall household pressure score out of 100`

Only the three core household indicators flow into the score. The additional context indicators are visual diagnostics, not score inputs.


## Custom Dashboard Layout

The exported dashboard is organized as a presentation page: current pressure status first, then the normalized pressure score, latest normalized pressure scores, the two full-history stress timelines, and finally an accordion section for the individual supporting plots.


In [94]:

from IPython.display import HTML

chart_sections = [
    ("Current Household Financial Pressure by Indicator", "Core household indicators ranked by their latest normalized pressure score, where 0 indicates low concern and 100 indicates high concern.", "normalized_financial_pressure_score.html", "compact"),
    ("Household Financial Pressure Scores Across All Indicators", "Latest normalized pressure scores across core and context indicators, showing which areas are contributing the most and least to household financial strain.", "latest_pressure_percentiles_core_context.html", "standard"),
    ("Housing Payment-to-Income Ratio", None, "housing_affordability.html", "housing"),
    ("Core Household Stress Flags Timeline", "Counts how many core household indicators reached unusually high-pressure readings relative to their own histories in each quarter. More flags mean more household stress signals were elevated at the same time.", "core_household_stress_flags_timeline.html", "timeline"),
    ("Expanded Household Pressure Flags Timeline", "Counts how many indicators across the full dashboard reached unusually high-pressure readings relative to their own histories in each quarter. More flags indicate broader financial strain across the household and economic environment.", "expanded_household_pressure_flags_timeline.html", "timeline"),
]

accordion_groups = [
    ("Core Household Indicator Time Series", [
        ("Debt Service Ratio", "debt_service_ratio.html"),
        ("Personal Saving Rate", "personal_saving_rate.html"),
        ("Credit Card Delinquency Rate", "credit_card_delinquency_rate.html"),
    ]),
    ("Economic Context Indicators", [
        ("Unemployment Rate", "unemployment_rate.html"),
        ("Federal Funds Rate", "federal_funds_rate.html"),
    ]),
    ("Additional Household Pressure Context", [
        ("CPI Year-over-Year Inflation", "inflation_cpi_yoy.html"),
        ("Real Median Weekly Earnings", "real_weekly_earnings.html"),
        ("Consumer Sentiment", "consumer_sentiment.html"),
    ]),
]

status_class = overall_status.replace("/", "-").replace(" ", "-")
card_descriptions = {
    "Debt Service Ratio": "Share of disposable income going to required household debt payments.",
    "Personal Saving Rate": "How much income households are saving after spending and taxes.",
    "Credit Card Delinquency Rate": "Share of credit card loans where borrowers are falling behind.",
}
core_status_cards = "".join(
    f'''<article class="metric-card">
        <span>{row["Metric"]}</span>
        <p>{card_descriptions[row["Metric"]]}</p>
        <strong>{row["Latest value"]}</strong>
        <em>Normalized pressure score: {row["Pressure percentile"]} as of {row["Latest date"]}</em>
    </article>'''
    for _, row in scorecard_display.iloc[1:].iterrows()
)

def chart_iframe(title, description, file_name, variant="standard"):
    note_html = f'<p class="chart-note">{description}</p>' if description else ''
    sync_attr = ' data-sync-group="pressure-flags"' if variant == "timeline" else ''
    return f'''<section class="chart-section {variant}">
        <h2>{title}</h2>
        {note_html}
        <iframe src="{file_name}" title="{title}" loading="lazy"{sync_attr}></iframe>
    </section>'''

def accordion_html(group_title, charts):
    chart_markup = "".join(
        f'''<div class="accordion-chart">
            <h3>{title}</h3>
            <iframe src="{file_name}" title="{title}" loading="lazy"></iframe>
        </div>'''
        for title, file_name in charts
    )
    return f'''<details class="accordion-item">
        <summary>{group_title}</summary>
        <div class="accordion-body">{chart_markup}</div>
    </details>'''

top_chart_html = "".join(chart_iframe(*frame) for frame in chart_sections)
accordion_markup = "".join(accordion_html(title, charts) for title, charts in accordion_groups)

dashboard_html = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>U.S. Household Financial Pressure Dashboard</title>
<style>
:root {{
    --ink: #222222;
    --muted: #606975;
    --line: #d9dde3;
    --paper: #f6f7f9;
    --panel: #ffffff;
    --teal: #2a9d8f;
    --accent-blue: #5b9bc8;
    --red: #c84f4f;
    --blue: #33658a;
}}
* {{ box-sizing: border-box; }}
body {{
    margin: 0;
    color: var(--ink);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
    background: var(--paper);
}}
main {{ width: min(1120px, calc(100% - 32px)); margin: 0 auto; padding: 24px 0 44px; }}
.status-panel {{
    border: 1px solid var(--line);
    border-left: 5px solid var(--accent-blue);
    background: var(--panel);
    padding: 22px;
}}
.status-label {{ margin: 0 0 6px; color: var(--muted); font-size: 13px; font-weight: 600; }}
.status-value {{ margin: 0; font-size: clamp(34px, 5vw, 48px); line-height: 1.05; font-weight: 700; }}
.status-pill {{ display: inline-block; margin-top: 12px; padding: 6px 10px; border: 1px solid #b9d5e8; background: #eef7fc; color: #22536a; font-size: 13px; font-weight: 700; }}
.status-score {{ margin: 12px 0 0; font-size: 15px; color: var(--muted); }}
.status-summary {{ display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 12px; margin-top: 20px; }}
.metric-card {{ border: 1px solid var(--line); background: #ffffff; padding: 14px; min-height: 122px; }}
.metric-card span {{ display: block; color: #2f3740; font-size: 13px; font-weight: 700; line-height: 1.35; }}
.metric-card p {{ margin: 7px 0 0; min-height: 42px; color: var(--muted); font-size: 13px; line-height: 1.4; }}
.metric-card strong {{ display: block; margin: 10px 0 5px; font-size: 28px; line-height: 1; }}
.metric-card em {{ color: var(--muted); font-size: 12px; font-style: normal; line-height: 1.4; }}
.chart-section, .accordion-panel {{ margin-top: 18px; border: 1px solid var(--line); background: var(--panel); padding: 16px; }}
.chart-section.standard, .chart-section.timeline {{ overflow-x: auto; -webkit-overflow-scrolling: touch; }}
h2 {{ margin: 0 0 12px; font-size: 20px; font-weight: 700; }}
.chart-note {{ margin: -4px 0 12px; color: var(--muted); font-size: 14px; line-height: 1.45; }}
h3 {{ margin: 0 0 10px; font-size: 16px; font-weight: 700; }}
iframe {{ width: 100%; border: 0; background: #ffffff; }}
.chart-section.compact iframe {{ height: 400px; }}
.chart-section.housing iframe {{ height: 360px; }}
.chart-section.standard iframe {{ height: 520px; }}
.chart-section.timeline iframe {{ height: 400px; }}
.chart-section.standard iframe, .chart-section.timeline iframe {{ min-width: 900px; }}
.accordion-panel > p {{ margin: -4px 0 16px; color: var(--muted); font-size: 14px; line-height: 1.5; }}
.accordion-item {{ border-top: 1px solid var(--line); background: #ffffff; }}
.accordion-item:last-child {{ border-bottom: 1px solid var(--line); }}
summary {{ cursor: pointer; padding: 14px 4px; font-size: 15px; font-weight: 700; line-height: 1.2; }}
.accordion-body {{ display: grid; gap: 18px; padding: 0 0 18px; }}
.accordion-chart {{ border: 1px solid #e6eaee; padding: 14px; background: #fff; }}
.accordion-chart iframe {{ height: 340px; }}
@media (max-width: 860px) {{
    .status-panel {{ padding: 18px; }}
    .status-summary {{ grid-template-columns: 1fr; margin-top: 18px; }}
    main {{ width: min(100% - 20px, 1180px); padding-top: 12px; }}
    .chart-section, .accordion-panel {{ padding: 12px; }}
    .chart-section.compact iframe, .chart-section.housing iframe, .accordion-chart iframe {{ height: 420px; }}
    .chart-section.standard iframe, .chart-section.timeline iframe {{ height: 470px; min-width: 900px; }}
}}
</style>
</head>
<body>
<main>
    <section class="status-panel">
        <div>
            <p class="status-label">Current household pressure status</p>
            <h1 class="status-value">{overall_status.title()}</h1>
            <span class="status-pill {status_class}">{overall_pressure_score:.1f} / 100</span>
            <p class="status-score">Core score through {latest_date_used:%Y-%m-%d}.</p>
        </div>
        <div class="status-summary">{core_status_cards}</div>
    </section>
    {top_chart_html}
    <section class="accordion-panel">
        <h2>All Other Indicator Plots</h2>
        <p>Open a section to inspect the individual source charts behind the score and context views.</p>
        {accordion_markup}
    </section>
</main>
<script>
(function() {
    window.addEventListener("message", event => {
        const message = event.data || {};
        if (message.type !== "pressure-flags-x-domain") return;
        if (!Array.isArray(message.domain) || message.domain.length !== 2) return;

        document.querySelectorAll('iframe[data-sync-group="pressure-flags"]').forEach(frame => {
            if (!frame.contentWindow || frame.contentWindow === event.source) return;
            frame.contentWindow.postMessage(message, "*");
        });
    });
})();
</script>
</body>
</html>'''

display(HTML(dashboard_html))


SyntaxError: f-string: expecting '=', or '!', or ':', or '}' (1763668613.py, line 159)


## Save Outputs

The notebook saves the main dashboard shell plus standalone chart files. The main dashboard uses the standalone files as embedded panels, including accordions for the supporting individual plots.


In [76]:

scorecard_display_for_output = scorecard_display.rename(columns={"Pressure percentile": "Normalized pressure score"})

scorecard_html = f'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><title>Household Financial Pressure Scorecard</title>
<style>
body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; margin: 32px; color: #202124; }}
h1 {{ font-size: 24px; margin-bottom: 6px; }}
p {{ max-width: 900px; line-height: 1.45; color: #4b5563; }}
table {{ border-collapse: collapse; min-width: 880px; margin-top: 18px; }}
th, td {{ border-bottom: 1px solid #d9dee3; padding: 10px 12px; text-align: left; }}
th {{ background: #f4f6f8; font-weight: 650; }}
tr:first-child td {{ font-weight: 650; background: #fbfcfd; }}
</style></head><body>
<h1>U.S. Household Financial Pressure Scorecard</h1>
<p>The overall score averages the three core household normalized pressure scores. The score uses each core series' latest available observation, so individual dates differ by release schedule.</p>
{scorecard_display_for_output.to_html(index=False, escape=False)}
</body></html>'''

selected_outputs = {
    "normalized_financial_pressure_score.html": pressure_score_chart,
    "latest_pressure_percentiles_core_context.html": latest_pressure_percentiles_chart,
    "core_household_stress_flags_timeline.html": core_stress_timeline_chart,
    "expanded_household_pressure_flags_timeline.html": expanded_stress_timeline_chart,
    "debt_service_ratio.html": debt_service_chart,
    "personal_saving_rate.html": saving_rate_chart,
    "credit_card_delinquency_rate.html": delinquency_chart,
    "unemployment_rate.html": unemployment_chart,
    "federal_funds_rate.html": fed_funds_chart,
    "inflation_cpi_yoy.html": inflation_cpi_yoy_chart,
    "real_weekly_earnings.html": real_weekly_earnings_chart,
    "consumer_sentiment.html": consumer_sentiment_chart,
    "housing_affordability.html": housing_affordability_chart,
}

(OUTPUT_DIR / "overview_scorecard.html").write_text(scorecard_html, encoding="utf-8")
(OUTPUT_DIR / "financial_pressure_dashboard.html").write_text(dashboard_html, encoding="utf-8")
for filename, chart in selected_outputs.items():
    chart.save(OUTPUT_DIR / filename)

print("Saved selected HTML outputs:")
for filename in ["financial_pressure_dashboard.html", "overview_scorecard.html", *selected_outputs.keys()]:
    print(OUTPUT_DIR / filename)


Saved selected HTML outputs:
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/financial_pressure_dashboard.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/overview_scorecard.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/normalized_financial_pressure_score.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/latest_pressure_percentiles_core_context.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/core_household_stress_flags_timeline.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/expanded_household_pressure_flags_timeline.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/debt_service_ratio.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/personal_saving_rate.html
/Users/bhawley/PycharmProjects/finacial-status-visualizer/html_charts/credit_card_delinquency_rate.html
/Users/bhawley/Pycharm


## Written Interpretation

The core household financial pressure score is **52.1 out of 100**, which is **moderate/mixed**. This score is unchanged and still uses only Debt Service Ratio, Personal Saving Rate, and Credit Card Delinquency Rate.

The expanded latest normalized pressure score chart includes context indicators, but it does **not** alter the core score. It shows each latest reading against that indicator's own history after adjusting direction so higher scores mean more pressure. The largest core pressure contributor remains **Personal Saving Rate** at **96.4**, while the least concerning core indicator remains **Debt Service Ratio** at **27.4**.

Context readings add background: CPI YoY inflation is **4.27%** as of **2026-05-01**; real median weekly earnings are **376** as of **2026-01-01**; consumer sentiment is **49.8** as of **2026-04-01**; and the estimated housing payment-to-income ratio is **28.0%** as of **2026-03-31**. These are context-only measures.

The core and expanded stress timelines now use the full available quarterly stress-window, displayed from the 1950s through the latest 2026 data. The core timeline still ranges from **0-3** and only counts the three core household indicators, but the tooltip shows how many core indicators were available in early quarters. The expanded timeline still ranges from **0-7** and shows available indicator count for the same reason.

The main HTML file to open is **`html_charts/financial_pressure_dashboard.html`**.
